# 2. Lookup decoder, MLP, and GRU

The lookup decoder uses only the latest observed syndrome. The MLP also sees only the latest round; the GRU sees all three rounds. Both learned models predict an X-component label and a Z-component label, each with eight classes (0 = no correction; 1–7 = reference qubit labels).

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import matplotlib.pyplot as plt
import torch
sys.path.insert(0, str(Path.cwd().resolve()))
from main import (load_split, MLPDecoder, GRUDecoder, train_model, predict_labels,
                  evaluate_predictions, lookup_predict, save_json)

train = load_split("data/train.npz")
validation = load_split("data/validation.npz")
test = load_split("data/test.npz")
y_test = np.stack([test["labels_x"], test["labels_z"]], axis=1)
print("Train trajectories:", len(train["observed"]), "Test trajectories:", len(test["observed"]))
print("Latest-round input shape:", train["observed"][:, -1, :].shape)
print("Sequence input shape:", train["observed"].shape)

In [ ]:
lookup_scores = evaluate_predictions(lookup_predict(test["observed"]), y_test)
print("Lookup baseline:", lookup_scores)

## Train the MLP

Adam computes gradients from cross-entropy loss and updates the network weights. After every epoch, the model is checked on the validation split; the checkpoint with the best exact-match validation accuracy is saved.

In [ ]:
torch.manual_seed(2026)
mlp, mlp_history = train_model(
    MLPDecoder(), "mlp", train, validation, "models/mlp.pt",
    epochs=25, batch_size=128, learning_rate=1e-3
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(mlp_history["train_loss"], label="train")
plt.plot(mlp_history["validation_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("MLP training history")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/mlp_loss.png", dpi=150)
plt.show()

## Train the GRU

The GRU reads the three rounds in order. Its hidden state carries a learned summary of earlier observations into later rounds, which may help it distinguish persistent syndrome information from measurement flips. This does not guarantee better performance.

In [ ]:
torch.manual_seed(2027)
gru, gru_history = train_model(
    GRUDecoder(hidden_size=24), "gru", train, validation, "models/gru.pt",
    epochs=25, batch_size=128, learning_rate=1e-3
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(gru_history["train_loss"], label="train")
plt.plot(gru_history["validation_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("GRU training history")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/gru_loss.png", dpi=150)
plt.show()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
mlp_checkpoint = torch.load("models/mlp.pt", map_location=device, weights_only=False)
gru_checkpoint = torch.load("models/gru.pt", map_location=device, weights_only=False)
mlp.load_state_dict(mlp_checkpoint["model_state_dict"])
gru.load_state_dict(gru_checkpoint["model_state_dict"])
mlp_features = test["observed"][:, -1, :].astype(np.float32)
gru_features = test["observed"].astype(np.float32)
mlp_scores = evaluate_predictions(predict_labels(mlp, mlp_features, device=device), y_test)
gru_scores = evaluate_predictions(predict_labels(gru, gru_features, device=device), y_test)
print("MLP test metrics:", mlp_scores)
print("GRU test metrics:", gru_scores)
save_json({
    "lookup": lookup_scores, "mlp": mlp_scores, "gru": gru_scores,
    "mlp_training_seconds": mlp_history["training_seconds"],
    "gru_training_seconds": gru_history["training_seconds"],
    "mlp_history": mlp_history, "gru_history": gru_history,
}, "results/classical_metrics.json")

## Summary

The lookup table is fixed. The MLP learns from one round; the GRU learns from the full sequence. The QNN notebook uses the same saved labels and splits.